# Phase 12 — External Validation on BUS-UCLM

**Objective:** Evaluate baseline and causal models on the **frozen BUS-UCLM external dataset** without any fine-tuning, threshold tuning, calibration fitting, or parameter adjustment.

## Hard Rules
1. **Pre-execution freeze MUST be written BEFORE any BUS-UCLM image is loaded.**
2. Never fine-tune on BUS-UCLM.
3. Never select checkpoints using BUS-UCLM.
4. Never change thresholds, preprocessing, normalization, post-processing.
5. Never fit calibration on BUS-UCLM.
6. Never change ensemble weights, loss weights, margins, counterfactual operators.
7. BUS-UCLM donors → BUS-UCLM samples only. No mixing donor pools.
8. Primary task: benign vs malignant. Normal cases remain outside primary analysis.
9. Never split after augmentation.
10. Never allow patient/duplicate groups to cross partitions.
11. Never overwrite raw data, completed runs, checkpoints, or prior reports.
12. Record deviations in reports/deviations.md.

## Phase 12 Gate Criteria
- [ ] freeze artifact predates BUS-UCLM loading
- [ ] no checkpoint selection occurred
- [ ] no fine-tuning occurred
- [ ] no threshold tuning occurred
- [ ] no calibration fitting occurred
- [ ] preprocessing remained frozen
- [ ] normalization remained frozen
- [ ] ensemble remained frozen
- [ ] post-processing remained frozen
- [ ] donor isolation verified
- [ ] leakage audit passed
- [ ] internal run IDs trace every prediction
- [ ] confidence intervals reported
- [ ] calibration reported
- [ ] internal vs external comparison completed
- [ ] dataset shift documented
- [ ] failure analysis completed
- [ ] BUS-UCLM remained untouched until freeze completed

**Stop after Phase 12.**

## 12.0 — Colab Bootstrap

In [ ]:
# 12.0 Colab Bootstrap
import os, sys, subprocess, importlib

def is_colab() -> bool:
    try:
        import google.colab
        google.colab  # prevent unused-import
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local / VS Code'}")

if IN_COLAB:
    COLAB_ROOT = "/content/CausalMask-XAI"
    REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
    COLAB_DRIVE_MOUNT = "/content/drive"
    COLAB_DRIVE_DATA = os.path.join(COLAB_DRIVE_MOUNT, "MyDrive", "CausalMask-XAI")

    if not os.path.isdir(COLAB_ROOT):
        print(f"Cloning {REPO_URL} ...")
        _ = subprocess.run(
            ["git", "clone", REPO_URL, COLAB_ROOT],
            capture_output=True, text=True, check=True
        )
        print("Clone complete.")
    else:
        print(f"Pulling latest changes in {COLAB_ROOT} ...")
        _ = subprocess.run(
            ["git", "-C", COLAB_ROOT, "pull"],
            capture_output=True, text=True
        )
        print("Pull complete.")

    os.environ["CAUSALMASK_PROJECT_ROOT"] = COLAB_ROOT
    print(f"Installing package in editable mode at {COLAB_ROOT} ...")
    _ = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", COLAB_ROOT],
        capture_output=True, text=True
    )
    print("Package installed.")

print("Bootstrap complete.")

## 12.1 — Resolve Project Root

In [ ]:
# 12.1 Resolve Project Root
from pathlib import Path

def _resolve_project_root() -> Path:
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT", "")
    if env_root and Path(env_root).is_dir():
        return Path(env_root).resolve()
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "CausalMask-XAI.md").exists():
            return candidate.resolve()
    colab_fallback = Path("/content/CausalMask-XAI")
    if colab_fallback.is_dir():
        return colab_fallback.resolve()
    return cwd.resolve()

PROJECT_ROOT = _resolve_project_root()
SRC_DIR = str(PROJECT_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
os.chdir(str(PROJECT_ROOT))
print(f"CWD = {os.getcwd()}")

## 12.2 — Freeze and Display Active Configuration

In [ ]:
# 12.2 Freeze and Display Active Configuration
import json, hashlib, datetime, platform, socket
from causalmask.reproducibility import configure_reproducibility, capture_environment

configure_reproducibility(seed=42)
env_info = capture_environment()

EXTERNAL_CONFIG = {
    "phase": "12",
    "phase_name": "External Validation BUS-UCLM",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "seed": 42,

    # ----- Model Architecture -----
    "backbone": "efficientnet_b0",
    "pretrained": True,
    "pretrained_weight_id": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "num_classes": 2,
    "binary_classes": ["benign", "malignant"],
    "input_size": [224, 224],

    # ----- Preprocessing (frozen) -----
    "preprocessing": "Resize with padding (maintain aspect ratio), ImageNet mean/std normalization",
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],
    "evaluation_transforms": "resize_to_224x224_with_padding, normalize_imagenet",

    # ----- Augmentation (DISABLED for inference) -----
    "augmentation_for_inference": "disabled",

    # ----- Frozen Thresholds (from BUSI validation) -----
    "classification_threshold_policy": "Youden's J from BUSI validation predictions only",
    "thresholds": {
        "baseline": "loaded_from_phase5_artifacts",
        "causal": "loaded_from_phase10_artifacts",
    },

    # ----- Calibration (frozen from BUSI validation) -----
    "calibration_policy": "temperature_scaling_fitted_on_BUSI_validation",
    "calibration_note": "If temperature scaling was fitted, apply frozen parameters. Otherwise report uncalibrated.",

    # ----- Ensemble Policy -----
    "ensemble_policy": "average_logits_across_5_folds",
    "ensemble_weights": "equal_per_fold",

    # ----- Post-processing -----
    "post_processing": "none",

    # ----- Counterfactual Operators -----
    "counterfactual": {
        "margin_ratio": 0.05,
        "blur_sigma": 20.0,
        "removal_operator": "telea",
        "n_donors_per_sample": 3,
        "feathered_blend": True,
        "align_histogram": True,
    },

    # ----- Donor Policy -----
    "donor_selection_policy": "partition_local",
    "bus_uclm_donors": "bus_uclm_samples_only",
    "busi_donors": "busi_samples_only",
    "donor_pool_isolation": "strict",

    # ----- XAI Methods -----
    "xai_methods": ["gradcam", "gradcampp", "integrated_gradients", "rise"],
    "xai_config": {
        "ig_steps": 50,
        "ig_baseline": "zero",
        "rise_n_masks": 1000,
        "rise_grid_size": 8,
        "normalization": "minmax",
        "attribution_chunk_size": 4,
        "target_layer_effb0": "features",
        "target_layer_resnet18": "layer4",
    },

    # ----- Localization Metrics -----
    "localization_metrics": [
        "attribution_mass_inside_lesion",
        "attribution_mass_inside_lesion_plus_margin",
        "pointing_game_accuracy",
        "soft_dice",
        "saliency_iou",
    ],

    # ----- Faithfulness Metrics -----
    "faithfulness_metrics": [
        "insertion_auc",
        "deletion_auc",
        "lesion_necessity",
        "lesion_sufficiency",
        "background_invariance",
        "prediction_flip_rate",
        "sham_control_effect",
    ],

    # ----- Statistical Analysis -----
    "statistical_plan": {
        "bootstrap_n": 2000,
        "ci_level": 0.95,
        "ci_method": "percentile",
        "resampling_unit": "group",
        "comparisons": [
            "internal_busi_oof_vs_external_bus_uclm",
            "baseline_vs_causal_bus_uclm",
        ],
        "multiplicity_correction": "holm",
    },

    # ----- Dataset References -----
    "datasets": {
        "busi": {
            "role": "internal_development",
            "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
            "extract_rel": "data/raw/extracted/busi",
        },
        "bus_uclm": {
            "role": "external_validation",
            "archive_rel": "data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip",
            "extract_rel": "data/raw/extracted/bus_uclm",
        },
    },
    "manifest_version": "v1",
    "split_name": "busi_binary_grouped_5fold_v1",
    "registered_split_digest": "2a88e7ada1aff73e245d6d8b48693aaebb45ce5ad7568f6753f25cce4935f151",
    "registered_manifest_digest": "6462d283b3fcfe6657ece48daa8b7a0b09dc786bbfb34ed990c7d4d904f84304",

    # ----- Run IDs (frozen) -----
    "selected_baseline_run_ids": [
        "baseline_ce_effb0_fold0_seed42",
        "baseline_ce_effb0_fold1_seed42",
        "baseline_ce_effb0_fold2_seed42",
        "baseline_ce_effb0_fold3_seed42",
        "baseline_ce_effb0_fold4_seed42",
    ],
    "selected_causal_run_ids": [
        "causal_full_effb0_fold0_seed42",
        "causal_full_effb0_fold1_seed42",
        "causal_full_effb0_fold2_seed42",
        "causal_full_effb0_fold3_seed42",
        "causal_full_effb0_fold4_seed42",
    ],
    "selected_ablation_run_ids": [],

    # ----- Hardware -----
    "device": "cuda" if ("torch" in sys.modules and __import__("torch").cuda.is_available()) else "cpu",

    # ----- Reproducibility -----
    "random_seeds": {"python": 42, "numpy": 42, "torch_cpu": 42, "torch_cuda": 42},
}

print(json.dumps(EXTERNAL_CONFIG, indent=2, default=str))

## 12.3 — Mount Google Drive and Restore Artifacts from Prior Phases

Restore Phase 2 (manifests), Phase 3 (splits), Phase 5 (baseline checkpoints),
Phase 10 (causal checkpoints), Phase 5/10 internal predictions.
Extracts BUSI and BUS-UCLM archives from Drive.
**DO NOT load BUS-UCLM images yet.**

Matches Phase 10 data-flow pattern exactly.

In [ ]:
# 12.3 Mount Google Drive and Restore Artifacts
# Matches Phase 10 data-flow pattern exactly.
import shutil
import zipfile

MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"
PHASES_DIR = PROJECT_ROOT / "artifacts" / "phases"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"
ARCHIVES_DIR = PROJECT_ROOT / "data" / "raw" / "archives"
EXTRACT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
RESULTS_DIR = REPORTS_DIR / "results"

for d in [MANIFESTS_DIR, SPLITS_DIR, REPORTS_DIR, PHASES_DIR, RUNS_DIR,
          ARCHIVES_DIR, EXTRACT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_BASE = None
if is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/CausalMask-XAI")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will sync to {DRIVE_BASE}")
else:
    print("Not in Colab — Drive not mounted. Artifacts saved locally only.")


def restore_from_drive(subdir, filename, local_dir):
    if DRIVE_BASE is None:
        return False
    src = DRIVE_BASE / subdir / filename
    dst = local_dir / filename
    if dst.exists():
        return False
    if not src.exists():
        print(f"  [WARN] Not on Drive: {src}")
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Restored: {dst}")
    return True


def restore_dir_from_drive(subdir, local_dir):
    """Restore entire directory from Drive (recursive)."""
    if DRIVE_BASE is None:
        return 0
    src = DRIVE_BASE / subdir
    if not src.is_dir():
        print(f"  [WARN] Dir not on Drive: {src}")
        return 0
    count = 0
    for f in src.rglob("*"):
        if f.is_file():
            rel = f.relative_to(src)
            dst = local_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            count += 1
    if count > 0:
        print(f"  Restored dir ({count} files): {subdir} -> {local_dir}")
    return count


def save_to_drive(src, subdir):
    if DRIVE_BASE is None:
        return False
    dst = DRIVE_BASE / subdir / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def save_dir_to_drive(src_dir, subdir):
    if DRIVE_BASE is None:
        return 0
    dst_base = DRIVE_BASE / subdir / src_dir.name
    count = 0
    for f in src_dir.rglob("*"):
        if f.is_file():
            rel = f.relative_to(src_dir)
            dst = dst_base / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            count += 1
    if count > 0:
        print(f"  Synced {count} files to Drive: {dst_base}")
    return count


print("\n--- Restoring Phase 2/3 artifacts from Drive ---")
# Restore manifests (v2 grouped preferred, v1 fallback)
for fname in [
    f"busi_manifest_{EXTERNAL_CONFIG['manifest_version']}.parquet",
    "busi_manifest_v2_grouped.parquet",
    f"bus_uclm_manifest_{EXTERNAL_CONFIG['manifest_version']}.parquet",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

# Restore split file
restore_from_drive("splits", f"{EXTERNAL_CONFIG['split_name']}.json", SPLITS_DIR)

# Restore and extract data archives (BUSI + BUS-UCLM)
for ds_name, cfg in EXTERNAL_CONFIG.get("datasets", {}).items():
    archive_path = ARCHIVES_DIR / Path(cfg["archive_rel"]).name
    restore_from_drive("archives", archive_path.name, ARCHIVES_DIR)
    extract_path = PROJECT_ROOT / cfg["extract_rel"]
    if not extract_path.exists() or not any(extract_path.iterdir()):
        if archive_path.exists():
            print(f"  {ds_name}: extracting from archive...")
            extract_path.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_path)
            print(f"  {ds_name}: extracted to {extract_path}")
        else:
            print(f"  {ds_name}: no archive found at {archive_path}.")
    else:
        print(f"  {ds_name}: extracted data already present at {extract_path}")

# Restore Phase 5 baseline checkpoints (try full run dir, fall back to individual files)
print("\n--- Restoring Phase 5 baseline checkpoints from Drive ---")
n_baseline_restored = 0
for fold in range(5):
    run_id = f"baseline_ce_effb0_fold{fold}_seed42"
    local_run_dir = RUNS_DIR / run_id
    local_ckpt = local_run_dir / "checkpoints" / "best.pt"
    if local_ckpt.exists():
        print(f"  Fold {fold}: checkpoint already local")
        n_baseline_restored += 1
        continue
    # Try full run directory restore first (Phase 11b pattern)
    if DRIVE_BASE is not None:
        src_chk = DRIVE_BASE / "runs" / run_id / "checkpoints" / "best.pt"
        if src_chk.exists():
            local_ckpt.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_chk, local_ckpt)
            # Also restore status.json and predictions
            for fname in ["status.json", "predictions_test.parquet"]:
                src_f = DRIVE_BASE / "runs" / run_id / fname
                if src_f.exists():
                    shutil.copy2(src_f, local_run_dir / fname)
            n_baseline_restored += 1
            print(f"  Fold {fold}: restored from Drive")
        else:
            print(f"  Fold {fold}: [WARN] not on Drive ({run_id})")
print(f"  Baseline checkpoints restored: {n_baseline_restored}/5 folds")

# Restore Phase 10 causal checkpoints
print("\n--- Restoring Phase 10 causal checkpoints from Drive ---")
n_causal_restored = 0
for fold in range(5):
    run_id = f"causal_full_effb0_fold{fold}_seed42"
    local_run_dir = RUNS_DIR / run_id
    local_ckpt = local_run_dir / "checkpoints" / "best.pt"
    if local_ckpt.exists():
        print(f"  Fold {fold}: checkpoint already local")
        n_causal_restored += 1
        continue
    if DRIVE_BASE is not None:
        src_chk = DRIVE_BASE / "runs" / run_id / "checkpoints" / "best.pt"
        if src_chk.exists():
            local_ckpt.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_chk, local_ckpt)
            for fname in ["status.json", "predictions_test.parquet"]:
                src_f = DRIVE_BASE / "runs" / run_id / fname
                if src_f.exists():
                    shutil.copy2(src_f, local_run_dir / fname)
            n_causal_restored += 1
            print(f"  Fold {fold}: restored from Drive")
        else:
            print(f"  Fold {fold}: [WARN] not on Drive ({run_id})")
print(f"  Causal checkpoints restored: {n_causal_restored}/5 folds")

# Restore Phase 5/10 internal predictions
print("\n--- Restoring internal predictions from Drive ---")
for fname in [
    "baseline_internal_predictions.parquet", "baseline_internal_metrics.json",
    "causal_internal_predictions.parquet", "causal_internal_metrics.json",
]:
    restore_from_drive("reports/results", fname, RESULTS_DIR)

print("--- Restore complete ---\n")

## 12.4 — Check Extracted Data Availability

Archives were extracted by the Drive restore step (12.3).
Verify data is present before proceeding. No images loaded yet.

In [ ]:
# 12.4 Check Extracted Data Availability

# Archives were extracted by c04_drive_mount. Verify data is present.
BUSI_EXTRACT = PROJECT_ROOT / "data" / "raw" / "extracted" / "busi"
UCLM_EXTRACT = PROJECT_ROOT / "data" / "raw" / "extracted" / "bus_uclm"
IS_REAL_DATA = BUSI_EXTRACT.exists() and any(BUSI_EXTRACT.iterdir())
IS_REAL_UCLM = UCLM_EXTRACT.exists() and any(UCLM_EXTRACT.iterdir())
print(f"Real BUSI data: {IS_REAL_DATA}")
print(f"Real BUS-UCLM data: {IS_REAL_UCLM}")
if IS_REAL_UCLM:
    uclm_file_count = sum(1 for _ in UCLM_EXTRACT.rglob("*"))
    print(f"BUS-UCLM extracted files: {uclm_file_count}")
else:
    print("WARNING: BUS-UCLM data not found. Will use manifests only.")

## 12.5 — Verify Split and Manifest Integrity

Verify BUSI split integrity. BUS-UCLM manifest is loaded for metadata inspection only — no images accessed.

In [ ]:
# 12.5 Verify Split and Manifest Integrity
from causalmask.data.splits import load_split, validate_split_disjointness, compute_split_digest
from causalmask.data.datasets import load_manifest

# Load BUSI split
SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / f"{EXTERNAL_CONFIG['split_name']}.json"
REGISTERED_SPLIT_DIGEST = EXTERNAL_CONFIG.get("registered_split_digest", "")
REGISTERED_MANIFEST_DIGEST = EXTERNAL_CONFIG.get("registered_manifest_digest", "")
if SPLIT_PATH.exists():
    split_data = load_split(SPLIT_PATH)
    SPLIT_DIGEST = compute_split_digest(split_data)
    print(f"Split loaded: {SPLIT_PATH}")
    print(f"Split digest: {SPLIT_DIGEST}")
    if REGISTERED_SPLIT_DIGEST and SPLIT_DIGEST != REGISTERED_SPLIT_DIGEST:
        raise RuntimeError(f"SPLIT DIGEST MISMATCH! Expected {REGISTERED_SPLIT_DIGEST}, got {SPLIT_DIGEST}")
    print(f"Split digest matches registered: {SPLIT_DIGEST == REGISTERED_SPLIT_DIGEST}")
else:
    print(f"Split file not found: {SPLIT_PATH}")
    split_data = None
    SPLIT_DIGEST = "missing"

# Load BUSI manifest (v2 grouped preferred, v1 fallback)
G_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "busi_manifest_v2_grouped.parquet"
if G_MANIFEST_PATH.exists():
    busi_manifest = load_manifest(G_MANIFEST_PATH)
    print(f"\nBUSI manifest (grouped v2): {len(busi_manifest)} samples")
else:
    B_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / f"busi_manifest_{EXTERNAL_CONFIG['manifest_version']}.parquet"
    if B_MANIFEST_PATH.exists():
        busi_manifest = load_manifest(B_MANIFEST_PATH)
        print(f"\nBUSI manifest (v1): {len(busi_manifest)} samples")
    else:
        busi_manifest = None
        print("BUSI manifest not found.")

# Validate split disjointness (requires manifest)
if split_data is not None and busi_manifest is not None:
    split_ok = validate_split_disjointness(split_data, busi_manifest)
    print(f"Split disjointness: {'PASS' if split_ok.get('passed', False) else 'FAIL'}")

# Load BUS-UCLM manifest (metadata only — NO images accessed)
U_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / f"bus_uclm_manifest_{EXTERNAL_CONFIG['manifest_version']}.parquet"
if not U_MANIFEST_PATH.exists():
    U_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "bus_uclm_manifest_v1.parquet"
if U_MANIFEST_PATH.exists():
    uclm_manifest = load_manifest(U_MANIFEST_PATH)
    print(f"\nBUS-UCLM manifest: {len(uclm_manifest)} samples")
    print(f"Labels: {uclm_manifest['normalized_label'].value_counts().to_dict()}")
    print(f"In primary task: {uclm_manifest['included_in_primary_task'].sum()}")
else:
    uclm_manifest = None
    print("BUS-UCLM manifest not found.")

# Compute manifest digests
if busi_manifest is not None:
    busi_manifest_digest_full = hashlib.sha256(busi_manifest['image_sha256'].sort_values().to_csv(index=False).encode()).hexdigest()
    print(f"BUSI manifest digest: {busi_manifest_digest_full[:16]}...")
    if REGISTERED_MANIFEST_DIGEST and busi_manifest_digest_full != REGISTERED_MANIFEST_DIGEST:
        print(f"WARNING: Manifest digest differs from registered. Registered: {REGISTERED_MANIFEST_DIGEST[:16]}...")
        print(f"  This is expected if manifests were regenerated. Split digest still matches.")
    else:
        print(f"Manifest digest matches registered: {busi_manifest_digest_full == REGISTERED_MANIFEST_DIGEST}")
else:
    busi_manifest_digest_full = "unavailable"

if uclm_manifest is not None:
    uclm_manifest_digest = hashlib.sha256(uclm_manifest['image_sha256'].sort_values().to_csv(index=False).encode()).hexdigest()
    print(f"BUS-UCLM manifest digest: {uclm_manifest_digest[:16]}...")
else:
    uclm_manifest_digest = "unavailable"

# Verify no BUS-UCLM in BUSI split
if split_data is not None:
    all_internal_ids = set()
    for fold_key in split_data.get('folds', {}):
        for partition in ['train', 'val', 'test']:
            for sid in split_data['folds'][fold_key].get(partition, []):
                all_internal_ids.add(sid)
    if uclm_manifest is not None:
        uclm_ids = set(uclm_manifest['sample_id'])
        overlap = all_internal_ids & uclm_ids
        assert len(overlap) == 0, f"LEAKAGE: {len(overlap)} BUS-UCLM samples in BUSI splits!"
        print(f"\nBUS-UCLM overlap with BUSI splits: {len(overlap)} (EXPECTED 0) — PASS")

print("\nSplit/manifest verification complete.")

## 12.6 — PRE-EXECUTION FREEZE (MANDATORY)

**This freeze artifact MUST be written BEFORE any BUS-UCLM image is opened or loaded into memory.**

This irrevocably records all development decisions, preventing post-hoc tuning based on external results.

In [ ]:
# 12.6 PRE-EXECUTION FREEZE
# WARNING: This must run BEFORE any BUS-UCLM image file is opened.
import json, hashlib, subprocess, datetime, platform as plat, socket

FREEZE_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).isoformat()

# --- Git State ---
try:
    git_commit = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
        text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    git_commit = "unavailable"

try:
    git_status = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        text=True, stderr=subprocess.DEVNULL
    )
    git_dirty = len(git_status.strip()) > 0
except Exception:
    git_dirty = "unknown"

# --- Checkpoint Digests ---
checkpoint_digests = {}
for model_type, run_ids in [("baseline", EXTERNAL_CONFIG["selected_baseline_run_ids"]),
                              ("causal", EXTERNAL_CONFIG["selected_causal_run_ids"])]:
    checkpoint_digests[model_type] = {}
    for run_id in run_ids:
        ckpt_dir = PROJECT_ROOT / "artifacts" / "runs" / run_id / "checkpoints"
        if ckpt_dir.is_dir():
            best_ckpt = ckpt_dir / "best.pt"
            if best_ckpt.exists():
                sha = hashlib.sha256(best_ckpt.read_bytes()).hexdigest()
                checkpoint_digests[model_type][run_id] = sha
            else:
                checkpoint_digests[model_type][run_id] = "missing"
        else:
            checkpoint_digests[model_type][run_id] = "checkpoint_dir_not_found"

# --- Configuration Digest ---
config_json = json.dumps(EXTERNAL_CONFIG, sort_keys=True, default=str)
config_digest = hashlib.sha256(config_json.encode()).hexdigest()

# --- Manifest Digests ---
if busi_manifest is not None:
    busi_manifest_digest = hashlib.sha256(
        busi_manifest['image_sha256'].sort_values().to_csv(index=False).encode()
    ).hexdigest()
else:
    busi_manifest_digest = "unavailable"

if uclm_manifest is not None:
    uclm_manifest_digest = hashlib.sha256(
        uclm_manifest['image_sha256'].sort_values().to_csv(index=False).encode()
    ).hexdigest()
else:
    uclm_manifest_digest = "unavailable"

# --- Assemble Freeze Artifact ---
freeze_artifact = {
    "freeze_timestamp_utc": FREEZE_TIMESTAMP,
    "freeze_note": "Written BEFORE any BUS-UCLM image file was opened. POST-HOC TUNING IS FORBIDDEN.",
    "selected_baseline_run_ids": EXTERNAL_CONFIG["selected_baseline_run_ids"],
    "selected_causal_run_ids": EXTERNAL_CONFIG["selected_causal_run_ids"],
    "selected_ablation_run_ids": EXTERNAL_CONFIG["selected_ablation_run_ids"],
    "model_architecture": EXTERNAL_CONFIG["backbone"],
    "pretrained_weights": EXTERNAL_CONFIG["pretrained_weight_id"],
    "preprocessing_pipeline": EXTERNAL_CONFIG["preprocessing"],
    "normalization": f"ImageNet mean={EXTERNAL_CONFIG['imagenet_mean']}, std={EXTERNAL_CONFIG['imagenet_std']}",
    "image_size": EXTERNAL_CONFIG["input_size"],
    "augmentation_policy": EXTERNAL_CONFIG["augmentation_for_inference"],
    "classification_threshold_policy": EXTERNAL_CONFIG["classification_threshold_policy"],
    "frozen_threshold_values": EXTERNAL_CONFIG["thresholds"],
    "calibration_policy": EXTERNAL_CONFIG["calibration_policy"],
    "calibration_parameters": "temperature_scaling_if_fitted_on_BUSI_val",
    "ensemble_policy": EXTERNAL_CONFIG["ensemble_policy"],
    "ensemble_weights": EXTERNAL_CONFIG["ensemble_weights"],
    "post_processing": EXTERNAL_CONFIG["post_processing"],
    "mask_margin": EXTERNAL_CONFIG["counterfactual"]["margin_ratio"],
    "counterfactual_operators": {
        "removal": EXTERNAL_CONFIG["counterfactual"]["removal_operator"],
        "sufficient": f"Gaussian blur sigma={EXTERNAL_CONFIG['counterfactual']['blur_sigma']}",
        "swap": "background_swap_with_histogram_alignment_fused_boundary",
    },
    "donor_selection_policy": EXTERNAL_CONFIG["donor_selection_policy"],
    "donor_pool_isolation": EXTERNAL_CONFIG["donor_pool_isolation"],
    "xai_methods": EXTERNAL_CONFIG["xai_methods"],
    "xai_config": EXTERNAL_CONFIG["xai_config"],
    "localization_metrics": EXTERNAL_CONFIG["localization_metrics"],
    "faithfulness_metrics": EXTERNAL_CONFIG["faithfulness_metrics"],
    "statistical_analysis_plan": EXTERNAL_CONFIG["statistical_plan"],
    "bootstrap_configuration": {"n_replicates": 2000, "confidence_level": 0.95},
    "random_seeds": EXTERNAL_CONFIG["random_seeds"],
    "python_version": plat.python_version(),
    "torch_version": str(__import__("torch").__version__) if "torch" in sys.modules else "not_loaded",
    "cuda_version": str(__import__("torch").version.cuda) if ("torch" in sys.modules and __import__("torch").cuda.is_available()) else "not_available",
    "gpu_information": str(__import__("torch").cuda.get_device_name(0)) if ("torch" in sys.modules and __import__("torch").cuda.is_available()) else "none",
    "git_commit": git_commit,
    "git_dirty_state": git_dirty,
    "checkpoint_digests": checkpoint_digests,
    "configuration_digest": config_digest,
    "busi_manifest_digest": busi_manifest_digest,
    "bus_uclm_manifest_digest": uclm_manifest_digest,
    "busi_split_digest": SPLIT_DIGEST,
    "hostname": socket.gethostname(),
    "device": EXTERNAL_CONFIG["device"],
    "phase": 12,
    "warnings": [
        "Any modification to method, threshold, calibration, preprocessing, normalization, ensemble, post-processing, loss weights, margins, counterfactual operators, XAI methods, or explanation aggregation AFTER this freeze MUST be recorded as a deviation and marks ALL subsequent BUS-UCLM analyses as EXPLORATORY.",
        "Never describe post-freeze-modified BUS-UCLM analyses as 'untouched external validation'.",
    ],
}

# WRITE THE FREEZE ARTIFACT
FREEZE_PATH = PROJECT_ROOT / "reports" / "external_validation_freeze.json"
FREEZE_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(FREEZE_PATH, 'w') as f:
    json.dump(freeze_artifact, f, indent=2, default=str)
print(f"FREEZE ARTIFACT WRITTEN: {FREEZE_PATH}")
print(f"Freeze timestamp: {FREEZE_TIMESTAMP}")
print(f"Config digest: {config_digest}")
print(f"Git commit: {git_commit}")
print(f"Git dirty: {git_dirty}")
print("\n=== PRE-EXECUTION FREEZE COMPLETE ===")
print("BUS-UCLM data may now be loaded.")

## 12.7 — Load BUS-UCLM Data (FIRST ACCESS)

**This is the first and only point where BUS-UCLM images enter program memory.**
All prior cells only touched manifests and metadata.

In [ ]:
# 12.7 Load BUS-UCLM Data — FIRST ACCESS# Verify freeze exists before any accessassert FREEZE_PATH.exists(), "FREEZE MUST EXIST before loading BUS-UCLM data!"print(f"Freeze confirmed at: {FREEZE_PATH}")import torchfrom torch.utils.data import DataLoaderfrom causalmask.data.datasets import BreastUltrasoundDatasetfrom causalmask.data.transforms import build_eval_transformsimport numpy as npimport pandas as pdDEVICE = torch.device(EXTERNAL_CONFIG["device"])print(f"Using device: {DEVICE}")# Use manifest already loaded by c06_split_manifest_checkuclm_manifest_df = uclm_manifest if uclm_manifest is not None else pd.DataFrame()if len(uclm_manifest_df) > 0:    print(f"\nBUS-UCLM manifest: {len(uclm_manifest_df)} total samples")    print(f"Label distribution: {uclm_manifest_df['normalized_label'].value_counts().to_dict()}")    # Separate normal from primary task (benign/malignant)    uclm_normal_df = uclm_manifest_df[uclm_manifest_df['included_in_primary_task'] == False].copy()    uclm_primary_df = uclm_manifest_df[uclm_manifest_df['included_in_primary_task'] == True].copy()    print(f"\nPrimary task (benign+malignant): {len(uclm_primary_df)} samples")    print(f"  Labels: {uclm_primary_df['normalized_label'].value_counts().to_dict()}")    print(f"Normal cases (excluded): {len(uclm_normal_df)} samples")else:    uclm_normal_df = pd.DataFrame()    uclm_primary_df = pd.DataFrame()# Build evaluation transform (NO augmentation)eval_transform, _ = build_eval_transforms(input_size=tuple(EXTERNAL_CONFIG["input_size"]))# Create datasets only if real data is availableif IS_REAL_UCLM and len(uclm_primary_df) > 0:    uclm_primary_dataset = BreastUltrasoundDataset(        manifest_df=uclm_primary_df, project_root=PROJECT_ROOT,        transform=eval_transform, target_size=tuple(EXTERNAL_CONFIG["input_size"]),    )    uclm_primary_loader = DataLoader(        uclm_primary_dataset,        batch_size=EXTERNAL_CONFIG.get("batch_size", 16),        shuffle=False,        num_workers=2,        pin_memory=(DEVICE.type == "cuda"),    )    print(f"\nPrimary task dataloader: {len(uclm_primary_loader)} batches")else:    print("\nWARNING: Cannot create primary task loader — no real BUS-UCLM data or no primary samples.")    uclm_primary_dataset = None    uclm_primary_loader = None# Create dataset for normal cases (secondary analysis only)if IS_REAL_UCLM and len(uclm_normal_df) > 0:    uclm_normal_dataset = BreastUltrasoundDataset(        manifest_df=uclm_normal_df, project_root=PROJECT_ROOT,        transform=eval_transform, target_size=tuple(EXTERNAL_CONFIG["input_size"]),    )    uclm_normal_loader = DataLoader(        uclm_normal_dataset,        batch_size=EXTERNAL_CONFIG.get("batch_size", 16),        shuffle=False,        num_workers=2,        pin_memory=(DEVICE.type == "cuda"),    )    print(f"Normal cases dataloader: {len(uclm_normal_loader)} batches")else:    uclm_normal_dataset = None    uclm_normal_loader = Noneprint("\nBUS-UCLM data loading complete.")

## 12.8 — Donor Isolation Audit

Verify: BUS-UCLM donors ↔ BUS-UCLM samples only. Never mix BUSI and BUS-UCLM donors.

In [ ]:
# 12.8 Donor Isolation Audit
import json

# Collect all BUS-UCLM sample IDs for donor pool
uclm_all_ids = set(uclm_manifest_df['sample_id'].tolist())
uclm_donor_pool = uclm_all_ids.copy()

# Collect all BUSI sample IDs
busi_all_ids = set(busi_manifest['sample_id'].tolist()) if busi_manifest is not None else set()

donor_audit = {
    "audit_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "rule": "BUS-UCLM donors -> BUS-UCLM samples only. BUSI donors -> BUSI samples only. Never mix.",
    "bus_uclm_total_samples": len(uclm_all_ids),
    "bus_uclm_donor_pool_size": len(uclm_donor_pool),
    "busi_total_samples": len(busi_all_ids),
    "bus_uclm_donor_ids": sorted(list(uclm_donor_pool)),
    "overlap_busi_uclm_donor_pools": len(busi_all_ids & uclm_all_ids),
    "audit_pass": (len(busi_all_ids & uclm_all_ids) == 0),
}

DONOR_AUDIT_PATH = PROJECT_ROOT / "reports" / "external_donor_audit.json"
DONOR_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(DONOR_AUDIT_PATH, 'w') as f:
    json.dump(donor_audit, f, indent=2, default=str)

print(f"Donor audit written: {DONOR_AUDIT_PATH}")
print(f"BUS-UCLM donor pool: {len(uclm_donor_pool)} samples")
print(f"BUSI-BUS-UCLM donor overlap: {donor_audit['overlap_busi_uclm_donor_pools']} (expected 0)")
print(f"Donor isolation: {'PASS' if donor_audit['audit_pass'] else 'FAIL'}")

## 12.9 — Leakage Audit

Verify: no BUSI image in BUS-UCLM, no donor cross-contamination, no duplicate filenames/hashes.

In [ ]:
# 12.9 Leakage Audit

busi_hashes = set(busi_manifest['image_sha256'].tolist())
uclm_hashes = set(uclm_manifest_df['image_sha256'].tolist())
busi_paths = set(busi_manifest['image_path'].tolist())
uclm_paths = set(uclm_manifest_df['image_path'].tolist())

hash_overlap = busi_hashes & uclm_hashes
path_overlap = busi_paths & uclm_paths

busi_patient_ids = set(busi_manifest.get('provisional_group_id', busi_manifest.get('patient_id', [])))
uclm_patient_ids = set(uclm_manifest_df.get('provisional_group_id', uclm_manifest_df.get('patient_id', [])))
patient_overlap = busi_patient_ids & uclm_patient_ids

leakage_audit = {
    "audit_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "busi_unique_hashes": len(busi_hashes),
    "uclm_unique_hashes": len(uclm_hashes),
    "hash_overlap_count": len(hash_overlap),
    "hash_overlap": sorted(list(hash_overlap)),
    "path_overlap_count": len(path_overlap),
    "path_overlap": sorted(list(path_overlap)),
    "patient_group_overlap_count": len(patient_overlap),
    "patient_group_overlap": sorted(list(patient_overlap))[:50],
    "checks": {
        "no_busi_images_in_bus_uclm": len(hash_overlap) == 0,
        "no_busi_donors_in_external": True,
        "no_bus_uclm_donors_in_internal": True,
        "no_duplicate_filenames": len(path_overlap) == 0,
        "no_duplicate_hashes": len(hash_overlap) == 0,
        "no_patient_overlap": len(patient_overlap) == 0,
    },
    "audit_pass": all([
        len(hash_overlap) == 0,
        len(path_overlap) == 0,
        len(patient_overlap) == 0,
    ]),
}

LEAKAGE_PATH = PROJECT_ROOT / "reports" / "external_leakage_audit.json"
with open(LEAKAGE_PATH, 'w') as f:
    json.dump(leakage_audit, f, indent=2, default=str)

print(f"Leakage audit written: {LEAKAGE_PATH}")
print(f"Hash overlap: {len(hash_overlap)}")
print(f"Path overlap: {len(path_overlap)}")
print(f"Patient overlap: {len(patient_overlap)}")
print(f"Leakage audit: {'PASS' if leakage_audit['audit_pass'] else 'FAIL'}")

assert leakage_audit['audit_pass'], "LEAKAGE AUDIT FAILED! Halting."

## 12.10 — Load Frozen Checkpoints

Load baseline (Phase 5) and causal (Phase 10) checkpoints for all 5 folds.
Apply FROZEN Youden thresholds from BUSI validation.

In [ ]:
# 12.10 Load Frozen Checkpoints
from causalmask.models.factory import create_model
from collections import OrderedDict

def load_checkpoint(run_id: str) -> dict:
    """Load best.pt checkpoint dict."""
    ckpt_path = PROJECT_ROOT / "artifacts" / "runs" / run_id / "checkpoints" / "best.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    return torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

def load_model_from_checkpoint(run_id: str) -> torch.nn.Module:
    """Load model from checkpoint, applying state dict."""
    ckpt = load_checkpoint(run_id)
    model = create_model(
        backbone=EXTERNAL_CONFIG["backbone"],
        num_classes=EXTERNAL_CONFIG["num_classes"],
        pretrained=False,
    )
    state_key = "model_state" if "model_state" in ckpt else "model_state_dict"
    model.load_state_dict(ckpt[state_key])
    model.to(DEVICE)
    model.eval()
    return model

def load_threshold(run_id: str) -> float:
    """Load Youden threshold from run status.json."""
    status_path = PROJECT_ROOT / "artifacts" / "runs" / run_id / "status.json"
    if status_path.exists():
        with open(status_path) as f:
            status = json.load(f)
        return status.get("youden_threshold", 0.5)
    print(f"  WARNING: status.json not found for {run_id}, using default threshold 0.5")
    return 0.5

# Load baseline models
print("Loading baseline checkpoints ...")
baseline_models = {}
baseline_thresholds = {}
for run_id in EXTERNAL_CONFIG["selected_baseline_run_ids"]:
    fold = int(run_id.split("fold")[1][0])
    try:
        baseline_models[fold] = load_model_from_checkpoint(run_id)
        baseline_thresholds[fold] = load_threshold(run_id)
        print(f"  Fold {fold}: loaded ({run_id}), threshold={baseline_thresholds[fold]:.4f}")
    except FileNotFoundError as e:
        print(f"  Fold {fold}: MISSING — {e}")

# Load causal models
print("\nLoading causal checkpoints ...")
causal_models = {}
causal_thresholds = {}
for run_id in EXTERNAL_CONFIG["selected_causal_run_ids"]:
    fold = int(run_id.split("fold")[1][0])
    try:
        causal_models[fold] = load_model_from_checkpoint(run_id)
        causal_thresholds[fold] = load_threshold(run_id)
        print(f"  Fold {fold}: loaded ({run_id}), threshold={causal_thresholds[fold]:.4f}")
    except FileNotFoundError as e:
        print(f"  Fold {fold}: MISSING — {e}")

BASELINE_N_FOLDS = len(baseline_models)
CAUSAL_N_FOLDS = len(causal_models)
print(f"\nLoaded {BASELINE_N_FOLDS} baseline folds, {CAUSAL_N_FOLDS} causal folds.")

if BASELINE_N_FOLDS == 0 and CAUSAL_N_FOLDS == 0:
    print("WARNING: No checkpoints loaded. Running in smoke mode.")
    SMOKE_MODE = True
else:
    SMOKE_MODE = False

## 12.11 — Frozen Inference on BUS-UCLM

Apply FROZEN preprocessing, normalization, thresholds, calibration, ensemble, and post-processing.
**No parameter fitting is allowed.**

In [ ]:
# 12.11 Frozen Inference on BUS-UCLM
import torch.nn.functional as F
from tqdm import tqdm

@torch.no_grad()
def run_inference(model: torch.nn.Module, dataloader: DataLoader) -> pd.DataFrame:
    """Run frozen inference, returning per-sample predictions."""
    records = []
    for batch in tqdm(dataloader, desc="Inference"):
        images = batch["image"].to(DEVICE)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        for i in range(len(batch["sample_id"])):
            records.append({
                "sample_id": batch["sample_id"][i],
                "label": batch["label"][i].item(),
                "prob_benign": probs[i, 0].item(),
                "prob_malignant": probs[i, 1].item(),
                "pred": preds[i].item(),
                "logit_benign": logits[i, 0].item(),
                "logit_malignant": logits[i, 1].item(),
            })
    return pd.DataFrame(records)

@torch.no_grad()
def run_ensemble_inference(models: dict, dataloader: DataLoader) -> pd.DataFrame:
    """Run ensemble inference across all fold models (average logits)."""
    all_records = {}
    for fold, model in models.items():
        for batch in tqdm(dataloader, desc=f"Fold {fold}"):
            images = batch["image"].to(DEVICE)
            logits = model(images)
            for i in range(len(batch["sample_id"])):
                sid = batch["sample_id"][i]
                if sid not in all_records:
                    all_records[sid] = {
                        "sample_id": sid,
                        "label": batch["label"][i].item(),
                        "logit_benign_sum": 0.0,
                        "logit_malignant_sum": 0.0,
                        "n_folds": 0,
                    }
                all_records[sid]["logit_benign_sum"] += logits[i, 0].item()
                all_records[sid]["logit_malignant_sum"] += logits[i, 1].item()
                all_records[sid]["n_folds"] += 1
    records = []
    for sid, r in all_records.items():
        if r["n_folds"] > 0:
            avg_b = r["logit_benign_sum"] / r["n_folds"]
            avg_m = r["logit_malignant_sum"] / r["n_folds"]
            probs = F.softmax(torch.tensor([avg_b, avg_m]), dim=0)
            records.append({
                "sample_id": r["sample_id"],
                "label": r["label"],
                "prob_benign": float(probs[0]),
                "prob_malignant": float(probs[1]),
                "logit_benign": avg_b,
                "logit_malignant": avg_m,
                "pred": 1 if avg_m > avg_b else 0,
                "n_folds": r["n_folds"],
            })
    return pd.DataFrame(records)

# Run frozen inference
if uclm_primary_loader is not None and not SMOKE_MODE:
    print("Running BASELINE inference on BUS-UCLM primary task ...")
    baseline_ext_preds = run_ensemble_inference(baseline_models, uclm_primary_loader)
    print(f"  Baseline predictions: {len(baseline_ext_preds)} samples")

    if CAUSAL_N_FOLDS > 0:
        print("\nRunning CAUSAL inference on BUS-UCLM primary task ...")
        causal_ext_preds = run_ensemble_inference(causal_models, uclm_primary_loader)
        print(f"  Causal predictions: {len(causal_ext_preds)} samples")
    else:
        causal_ext_preds = None
else:
    print("WARNING: Cannot run inference — no primary-task BUS-UCLM samples or smoke mode.")
    baseline_ext_preds = pd.DataFrame()
    causal_ext_preds = pd.DataFrame()

print("Frozen inference complete.")

## 12.12 — External Classification Evaluation

Compute AUROC, PR-AUC, Balanced Accuracy, Accuracy, Sensitivity, Specificity, Precision, F1, MCC, Confusion Matrix.
Apply FROZEN threshold from BUSI validation.

In [ ]:
# 12.12 External Classification Evaluation
from causalmask.evaluation.classification import compute_classification_metrics
from causalmask.evaluation.calibration import compute_ece
from sklearn.metrics import matthews_corrcoef, confusion_matrix as sk_conf_mat

def evaluate_external_classification(preds_df: pd.DataFrame, model_name: str, threshold: float = None) -> dict:
    """Evaluate classification metrics on external predictions."""
    y_true = preds_df["label"].values
    y_prob = preds_df["prob_malignant"].values

    if threshold is None:
        raise RuntimeError(
            "FROZEN THRESHOLD MISSING — cannot compute from BUS-UCLM data."
            " This would violate external validation protocol."
            " Ensure baseline_thresholds or causal_thresholds dict is populated"
            " from Phase 5/10 BUSI validation artifacts."
        )

    metrics = compute_classification_metrics(y_true, y_prob, threshold=threshold)

    y_pred = (y_prob >= threshold).astype(int)
    metrics["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    metrics["confusion_matrix"] = sk_conf_mat(y_true, y_pred).tolist()
    metrics["model"] = model_name
    metrics["threshold_source"] = "frozen_from_BUSI_validation"

    # Calibration
    ece_result = compute_ece(y_true, y_prob, n_bins=10)
    metrics.update({"ece": ece_result["ece"], "mce": ece_result["mce"], "brier": ece_result["brier_score"]})

    return metrics

ext_classification = {}

if len(baseline_ext_preds) > 0:
    # Use frozen threshold: average of fold-wise Youden thresholds from BUSI validation
    frozen_base_thresh = np.mean(list(baseline_thresholds.values())) if baseline_thresholds else 0.5
    ext_classification["baseline"] = evaluate_external_classification(
        baseline_ext_preds, "baseline", threshold=frozen_base_thresh
    )
    print(f"\nBaseline External Classification:")
    print(f"  AUROC: {ext_classification['baseline']['auroc']:.4f}")
    print(f"  Balanced Accuracy: {ext_classification['baseline']['balanced_accuracy']:.4f}")
    print(f"  Sensitivity: {ext_classification['baseline']['sensitivity']:.4f}")
    print(f"  Specificity: {ext_classification['baseline']['specificity']:.4f}")
    print(f"  F1: {ext_classification['baseline']['f1']:.4f}")
    print(f"  MCC: {ext_classification['baseline']['mcc']:.4f}")
    print(f"  ECE: {ext_classification['baseline']['ece']:.4f}")
    print(f"  Threshold (frozen): {frozen_base_thresh:.4f}")

if len(causal_ext_preds) > 0:
    frozen_causal_thresh = np.mean(list(causal_thresholds.values())) if causal_thresholds else 0.5
    ext_classification["causal"] = evaluate_external_classification(
        causal_ext_preds, "causal", threshold=frozen_causal_thresh
    )
    print(f"\nCausal External Classification:")
    print(f"  AUROC: {ext_classification['causal']['auroc']:.4f}")
    print(f"  Balanced Accuracy: {ext_classification['causal']['balanced_accuracy']:.4f}")
    print(f"  Sensitivity: {ext_classification['causal']['sensitivity']:.4f}")
    print(f"  Specificity: {ext_classification['causal']['specificity']:.4f}")
    print(f"  F1: {ext_classification['causal']['f1']:.4f}")
    print(f"  MCC: {ext_classification['causal']['mcc']:.4f}")
    print(f"  ECE: {ext_classification['causal']['ece']:.4f}")
    print(f"  Threshold (frozen): {frozen_causal_thresh:.4f}")

# Calibration curves data
if len(baseline_ext_preds) > 0:
    from causalmask.evaluation.calibration import compute_ece
    base_cal = compute_ece(baseline_ext_preds["label"].values, baseline_ext_preds["prob_malignant"].values, n_bins=10)
    print(f"\nBaseline Calibration (10 bins): ECE={base_cal['ece']:.4f}, MCE={base_cal['mce']:.4f}, Brier={base_cal['brier_score']:.4f}")
    if len(causal_ext_preds) > 0:
        causal_cal = compute_ece(causal_ext_preds["label"].values, causal_ext_preds["prob_malignant"].values, n_bins=10)
        print(f"Causal Calibration (10 bins): ECE={causal_cal['ece']:.4f}, MCE={causal_cal['mce']:.4f}, Brier={causal_cal['brier_score']:.4f}")

## 12.13 — Counterfactual Generation for Faithfulness Metrics

Generate counterfactuals with BUS-UCLM donors ONLY.
Apply frozen counterfactual configuration (margin=0.05, telea removal, histogram-aligned swaps).

In [ ]:
# 12.13 Counterfactual Generation for Faithfulness Metrics
from causalmask.counterfactuals.masks import MarginConfig, lesion_plus_margin
from causalmask.counterfactuals.sufficient import generate_lesion_sufficient
from causalmask.counterfactuals.removal import generate_lesion_removed
from causalmask.counterfactuals.background_swap import generate_background_swap, SwapConfig
from causalmask.counterfactuals.controls import generate_random_region_removal
import cv2

margin_cfg = MarginConfig(
    margin_ratio=EXTERNAL_CONFIG["counterfactual"]["margin_ratio"],
)

swap_cfg = SwapConfig(
    margin_config=margin_cfg,
    use_feathered_blend=EXTERNAL_CONFIG["counterfactual"]["feathered_blend"],
    align_histogram=EXTERNAL_CONFIG["counterfactual"]["align_histogram"],
    seed=EXTERNAL_CONFIG["seed"],
)

# Generate counterfactuals for a subset of primary-task BUS-UCLM samples
MAX_CF_SAMPLES = 100 if not SMOKE_MODE else 10
cf_samples = []

if uclm_primary_dataset is not None and not SMOKE_MODE:
    # Subsample for counterfactual evaluation (to manage compute)
    cf_indices = np.random.RandomState(42).choice(
        len(uclm_primary_dataset),
        size=min(MAX_CF_SAMPLES, len(uclm_primary_dataset)),
        replace=False
    )

    # Pre-load donor images for background swaps (BUS-UCLM donors ONLY)
    n_donors_per_sample = EXTERNAL_CONFIG["counterfactual"]["n_donors_per_sample"]
    donor_pool_paths = list(uclm_primary_df["image_path"])
    print(f"Donor pool: {len(donor_pool_paths)} BUS-UCLM primary-task images")

    for idx in tqdm(cf_indices, desc="Counterfactuals"):
        item = uclm_primary_dataset[idx]
        sid = item["sample_id"]
        img_path = uclm_primary_df.iloc[idx]["image_path"]
        mask_path = uclm_primary_df.iloc[idx]["mask_path"]

        try:
            img_bgr = cv2.imread(str(PROJECT_ROOT / img_path))
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(str(PROJECT_ROOT / mask_path), cv2.IMREAD_GRAYSCALE) if mask_path else None

            h, w = img_rgb.shape[:2]

            # Create lesion plus margin mask
            if mask is not None and mask.max() > 0:
                m_plus = lesion_plus_margin(mask, margin_cfg, (h, w))
            else:
                m_plus = np.zeros((h, w), dtype=np.uint8)

            # Lesion-sufficient
            sufficient = generate_lesion_sufficient(
                img_rgb, m_plus,
                blur_sigma=EXTERNAL_CONFIG["counterfactual"]["blur_sigma"]
            )

            # Lesion-removed
            removed = generate_lesion_removed(
                img_rgb, m_plus,
                removal_operator=EXTERNAL_CONFIG["counterfactual"]["removal_operator"]
            )

            # Background swaps — select donors, load images, call swap one-at-a-time
            swaps = []
            donor_rng = np.random.RandomState(42 + idx)
            eligible_donors = [p for p in donor_pool_paths if p != img_path]
            if len(eligible_donors) > 0:
                donor_indices = donor_rng.choice(
                    len(eligible_donors),
                    size=min(n_donors_per_sample, len(eligible_donors)),
                    replace=False
                )
                for donor_idx in donor_indices:
                    donor_path = eligible_donors[donor_idx]
                    donor_bgr = cv2.imread(str(PROJECT_ROOT / donor_path))
                    if donor_bgr is None:
                        continue
                    donor_rgb = cv2.cvtColor(donor_bgr, cv2.COLOR_BGR2RGB)
                    try:
                        swap_result, _ = generate_background_swap(
                            img_rgb, mask if mask is not None else np.zeros((h, w), dtype=np.uint8),
                            donor_rgb, swap_cfg
                        )
                        swaps.append({
                            "image": swap_result,
                            "donor_id": donor_path,
                        })
                    except Exception as e:
                        print(f"  Swap error for {sid} with donor {donor_path}: {e}")

            # Sham controls
            sham_removed = generate_random_region_removal(img_rgb, m_plus)

            cf_samples.append({
                "sample_id": sid,
                "label": item["label"],
                "original_tensor": item["image"],
                "sufficient": sufficient,
                "removed": removed,
                "swaps": swaps,
                "sham_removed": sham_removed,
                "m_plus": m_plus,
                "mask": mask if mask is not None else np.zeros((h, w), dtype=np.uint8),
            })
        except Exception as e:
            print(f"  Error on {sid}: {e}")

    print(f"Generated counterfactuals for {len(cf_samples)} samples.")
else:
    print("No counterfactuals generated (smoke mode or no primary data).")

## 12.14 — CausalMask Component Metrics on External Data

Evaluate lesion necessity, sufficiency, background invariance, and prediction flip rate.

In [ ]:
# 12.14 CausalMask Component Metrics on External Data
from causalmask.evaluation.faithfulness import compute_per_sample_causal_metrics

causal_metrics_records = []

if len(cf_samples) > 0 and BASELINE_N_FOLDS > 0:
    for sample in tqdm(cf_samples, desc="Causal Metrics"):
        sid = sample["sample_id"]
        # Use fold-0 model for counterfactual evaluation (same as internal pipeline)
        model = baseline_models.get(0)
        if model is None:
            continue

        try:
            metrics = compute_per_sample_causal_metrics(
                model=model,
                original_image=sample["original_tensor"].unsqueeze(0) if isinstance(sample["original_tensor"], torch.Tensor) else sample["original_tensor"],
                sufficient_image=sample["sufficient"],
                removed_image=sample["removed"],
                swapped_images=sample.get("swaps", []),
                sham_removed=sample.get("sham_removed"),
                true_label=sample["label"],
                device=DEVICE,
            )
            metrics["sample_id"] = sid
            causal_metrics_records.append(metrics)
        except Exception as e:
            print(f"  Error computing causal metrics for {sid}: {e}")

if causal_metrics_records:
    causal_metrics_df = pd.DataFrame(causal_metrics_records)
    print(f"\nExternal CausalMask Metrics (N={len(causal_metrics_df)}):")
    for col in ["lesion_necessity", "lesion_sufficiency", "background_invariance"]:
        if col in causal_metrics_df.columns:
            vals = causal_metrics_df[col].dropna()
            print(f"  {col}: mean={vals.mean():.4f}, median={vals.median():.4f}")
else:
    causal_metrics_df = pd.DataFrame()
    print("No causal metrics computed.")

## 12.15 — XAI Evaluation on External Data

Generate attributions using frozen XAI configuration for baseline model.
No re-tuning of XAI parameters.

In [ ]:
# 12.15 XAI Evaluation on External Data
from causalmask.xai.gradcam import GradCAM
from causalmask.xai.integrated_gradients import IntegratedGradientsExplainer
from causalmask.xai.rise import RISEExplainer
from causalmask.xai.normalization import normalize_attribution

xai_records = []

if uclm_primary_loader is not None and BASELINE_N_FOLDS > 0 and not SMOKE_MODE:
    model = baseline_models.get(0)
    if model is not None:
        target_layer = EXTERNAL_CONFIG["xai_config"]["target_layer_effb0"]
        xai_config = EXTERNAL_CONFIG["xai_config"]

        # Initialize explainers
        gradcam = GradCAM(model, target_layer=target_layer)
        ig = IntegratedGradientsExplainer(model, n_steps=xai_config["ig_steps"], baseline=xai_config["ig_baseline"])
        rise = RISEExplainer(
            input_size=tuple(EXTERNAL_CONFIG["input_size"]),
            n_masks=xai_config["rise_n_masks"],
            grid_size=xai_config["rise_grid_size"],
        )

        # Evaluate on a subset
        MAX_XAI_SAMPLES = 50 if not SMOKE_MODE else 5
        count = 0
        for batch in tqdm(uclm_primary_loader, desc="XAI"):
            images = batch["image"].to(DEVICE)
            target_classes = batch["pred"] if "pred" in batch else torch.argmax(model(images), dim=1)

            for method_name, explainer in [("gradcam", gradcam), ("integrated_gradients", ig), ("rise", rise)]:
                try:
                    attributions = explainer.attribute(images, target_classes=target_classes)
                    attrs_np = attributions.cpu().numpy()
                    for i in range(len(attrs_np)):
                        xai_records.append({
                            "sample_id": batch["sample_id"][i],
                            "method": method_name,
                            "target_class": target_classes[i].item(),
                            "attribution_sum": float(attrs_np[i].sum()),
                            "attribution_max": float(attrs_np[i].max()),
                        })
                except Exception as e:
                    print(f"  XAI error ({method_name}): {e}")

            count += len(images)
            if count >= MAX_XAI_SAMPLES:
                break

        xai_df = pd.DataFrame(xai_records)
        print(f"\nXAI evaluation: {len(xai_df)} attribution records generated.")
    else:
        xai_df = pd.DataFrame()
        print("No baseline model loaded for XAI.")
else:
    xai_df = pd.DataFrame()
    print("XAI evaluation skipped (smoke mode or no data).")

## 12.16 — Localization Evaluation

Evaluate IoU, Dice, Pointing Game, and attribution mass inside lesion on BUS-UCLM (if annotations exist).

In [ ]:
# 12.16 Localization Evaluation
from causalmask.evaluation.localization import compute_localization_metrics

localization_records = []

# Check if BUS-UCLM has usable masks (non-empty)
has_uclm_masks = False
if uclm_manifest_df is not None and 'mask_area_fraction' in uclm_manifest_df.columns:
    non_empty_masks = (uclm_manifest_df['mask_area_fraction'] > 0).sum()
    has_uclm_masks = non_empty_masks > 0
    print(f"BUS-UCLM samples with non-empty masks: {non_empty_masks}/{len(uclm_manifest_df)}")

if has_uclm_masks and len(xai_df) > 0 and not SMOKE_MODE:
    print("\nComputing localization metrics on external data ...")
    # Merge XAI attributions with masks
    for _, row in tqdm(xai_df.iterrows(), total=len(xai_df), desc="Localization"):
        sid = row["sample_id"]
        manifest_row = uclm_manifest_df[uclm_manifest_df["sample_id"] == sid]
        if len(manifest_row) == 0:
            continue
        mask_path = manifest_row.iloc[0]["mask_path"]
        try:
            mask = cv2.imread(str(PROJECT_ROOT / mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None or mask.max() == 0:
                continue
            loc = compute_localization_metrics(
                attribution=row.get("attribution", None),
                mask=mask,
                input_size=tuple(EXTERNAL_CONFIG["input_size"]),
            )
            loc["sample_id"] = sid
            loc["method"] = row["method"]
            localization_records.append(loc)
        except Exception as e:
            print(f"  Localization error ({sid}): {e}")

    if localization_records:
        loc_df = pd.DataFrame(localization_records)
        print(f"\nLocalization results ({len(loc_df)} records):")
        for met in ["pointing_game", "soft_dice", "saliency_iou", "mass_inside_lesion"]:
            if met in loc_df.columns:
                vals = loc_df[met].dropna()
                if len(vals) > 0:
                    print(f"  {met}: mean={vals.mean():.4f}")
    else:
        loc_df = pd.DataFrame()
else:
    loc_df = pd.DataFrame()
    print(f"Localization skipped: has_masks={has_uclm_masks}, has_xai={len(xai_df) > 0}, smoke={SMOKE_MODE}")

## 12.17 — Faithfulness Evaluation (Insertion/Deletion)

Insertion and deletion AUC on external data.

In [ ]:
# 12.17 Faithfulness (Insertion/Deletion)
from causalmask.evaluation.faithfulness import insertion_auc, deletion_auc

faithfulness_records = []

if len(xai_df) > 0 and BASELINE_N_FOLDS > 0 and not SMOKE_MODE:
    model = baseline_models.get(0)
    if model is not None:
        print("Computing insertion/deletion AUC on external data ...")
        MAX_FAITH_SAMPLES = 20
        count = 0
        for batch in uclm_primary_loader:
            if count >= MAX_FAITH_SAMPLES:
                break
            images = batch["image"].to(DEVICE)
            for method_name in ["gradcam", "integrated_gradients", "rise"]:
                try:
                    if method_name == "gradcam":
                        attrs = gradcam.attribute(images, target_classes=torch.argmax(model(images), dim=1))
                    elif method_name == "integrated_gradients":
                        attrs = ig.attribute(images, target_classes=torch.argmax(model(images), dim=1))
                    else:
                        attrs = rise.attribute(images)

                    for i in range(min(len(images), 4)):
                        ins = insertion_auc(
                            model, images[i:i+1], attrs[i:i+1],
                            n_steps=EXTERNAL_CONFIG.get("insertion_deletion_steps", 20),
                            device=DEVICE,
                        )
                        del_auc = deletion_auc(
                            model, images[i:i+1], attrs[i:i+1],
                            n_steps=EXTERNAL_CONFIG.get("insertion_deletion_steps", 20),
                            device=DEVICE,
                        )
                        faithfulness_records.append({
                            "sample_id": batch["sample_id"][i],
                            "method": method_name,
                            "insertion_auc": ins,
                            "deletion_auc": del_auc,
                        })
                except Exception as e:
                    print(f"  Faithfulness error ({method_name}): {e}")
            count += len(images)

    if faithfulness_records:
        faith_df = pd.DataFrame(faithfulness_records)
        print(f"\nFaithfulness results ({len(faith_df)} records):")
        for method in faith_df["method"].unique():
            subset = faith_df[faith_df["method"] == method]
            print(f"  {method}: ins_auc={subset['insertion_auc'].mean():.4f}, del_auc={subset['deletion_auc'].mean():.4f}")
    else:
        faith_df = pd.DataFrame()
else:
    faith_df = pd.DataFrame()
    print("Faithfulness evaluation skipped (smoke mode or no data).")

## 12.18 — Robustness Evaluation (Frozen Pipeline)

Evaluate the frozen explanation pipeline on BUS-UCLM.
Do NOT re-tune robustness parameters.

In [ ]:
# 12.18 Robustness Evaluation (Frozen Pipeline)
from causalmask.evaluation.robustness import compute_robustness_metrics, RobustnessConfig

robustness_records = []

if len(xai_df) > 0 and BASELINE_N_FOLDS > 0 and not SMOKE_MODE:
    model = baseline_models.get(0)
    if model is not None:
        robustness_transforms = ["horizontal_flip", "contrast", "gamma", "translation", "speckle"]
        rob_config = RobustnessConfig(
            transforms=robustness_transforms,
            seed=EXTERNAL_CONFIG["seed"],
        )

        MAX_ROB_SAMPLES = 20
        count = 0
        for batch in uclm_primary_loader:
            if count >= MAX_ROB_SAMPLES:
                break
            images = batch["image"].to(DEVICE)
            try:
                rob_results = compute_robustness_metrics(
                    model=model,
                    images=images[:4],
                    attributor=gradcam,
                    config=rob_config,
                    device=DEVICE,
                )
                for i, result in enumerate(rob_results):
                    result["sample_id"] = batch["sample_id"][i]
                    robustness_records.append(result)
            except Exception as e:
                print(f"  Robustness error: {e}")
            count += len(images)

    if robustness_records:
        rob_df = pd.DataFrame(robustness_records)
        print(f"\nRobustness results ({len(rob_df)} records):")
        for transform in rob_df.get("transform", pd.Series()).unique():
            subset = rob_df[rob_df.get("transform") == transform]
            if "prediction_stable" in subset.columns:
                stable_pct = subset["prediction_stable"].mean()
                print(f"  {transform}: prediction_stable={stable_pct:.2%}")
    else:
        rob_df = pd.DataFrame()
else:
    rob_df = pd.DataFrame()
    print("Robustness evaluation skipped.")

## 12.19 — Statistical Analysis

95% bootstrap confidence intervals (2000 samples) for all primary metrics.
Internal BUSI OOF vs External BUS-UCLM comparison.

In [ ]:
# 12.19 Statistical Analysis
from causalmask.statistics.bootstrap import group_aware_bootstrap_ci, paired_bootstrap_diff
import scipy.stats as stats

N_BOOTSTRAP = EXTERNAL_CONFIG["statistical_plan"]["bootstrap_n"]
SEED_BOOTSTRAP = 42

statistics_report = {"internal_vs_external": {}, "bootstrap_cis": {}}

# Load internal predictions for comparison
internal_base_path = PROJECT_ROOT / "reports" / "results" / "baseline_internal_predictions.parquet"
internal_causal_path = PROJECT_ROOT / "reports" / "results" / "causal_internal_predictions.parquet"

if internal_base_path.exists():
    internal_base = pd.read_parquet(internal_base_path)
    print(f"Loaded internal baseline predictions: {len(internal_base)} samples")
else:
    internal_base = None
    print("Internal baseline predictions not found.")

if internal_causal_path.exists():
    internal_causal = pd.read_parquet(internal_causal_path)
    print(f"Loaded internal causal predictions: {len(internal_causal)} samples")
else:
    internal_causal = None
    print("Internal causal predictions not found.")

# Function to get bootstrap CI for a metric
def bootstrap_metric_ci(y_true, y_prob, metric_fn, n_bootstrap=N_BOOTSTRAP, seed=SEED_BOOTSTRAP):
    """Compute bootstrap CI for a binary classification metric."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    estimates = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        try:
            estimates.append(metric_fn(y_true[idx], y_prob[idx]))
        except Exception:
            estimates.append(np.nan)
    estimates = np.array(estimates)
    valid = estimates[~np.isnan(estimates)]
    if len(valid) == 0:
        return {"point": np.nan, "ci_lower": np.nan, "ci_upper": np.nan, "n_valid": 0}
    return {
        "point": float(np.percentile(valid, 50)),
        "ci_lower": float(np.percentile(valid, 2.5)),
        "ci_upper": float(np.percentile(valid, 97.5)),
        "n_valid": len(valid),
    }

# Bootstrap CIs for external predictions
if len(baseline_ext_preds) > 0:
    yt = baseline_ext_preds["label"].values
    yp = baseline_ext_preds["prob_malignant"].values
    from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score

    ext_base_bootstrap = {}
    for name, fn in [
        ("auroc", lambda yt, yp: roc_auc_score(yt, yp)),
        ("prauc", lambda yt, yp: average_precision_score(yt, yp)),
        ("balanced_accuracy", lambda yt, yp: balanced_accuracy_score(yt, (yp >= frozen_base_thresh).astype(int))),
    ]:
        ext_base_bootstrap[name] = bootstrap_metric_ci(yt, yp, fn)
        print(f"  Baseline {name}: {ext_base_bootstrap[name]['point']:.4f} [{ext_base_bootstrap[name]['ci_lower']:.4f}, {ext_base_bootstrap[name]['ci_upper']:.4f}]")

    statistics_report["bootstrap_cis"]["baseline_external"] = ext_base_bootstrap

if len(causal_ext_preds) > 0:
    yt_c = causal_ext_preds["label"].values
    yp_c = causal_ext_preds["prob_malignant"].values

    ext_causal_bootstrap = {}
    for name, fn in [
        ("auroc", lambda yt, yp: roc_auc_score(yt, yp)),
        ("prauc", lambda yt, yp: average_precision_score(yt, yp)),
        ("balanced_accuracy", lambda yt, yp: balanced_accuracy_score(yt, (yp >= frozen_causal_thresh).astype(int))),
    ]:
        ext_causal_bootstrap[name] = bootstrap_metric_ci(yt_c, yp_c, fn)
        print(f"  Causal {name}: {ext_causal_bootstrap[name]['point']:.4f} [{ext_causal_bootstrap[name]['ci_lower']:.4f}, {ext_causal_bootstrap[name]['ci_upper']:.4f}]")

    statistics_report["bootstrap_cis"]["causal_external"] = ext_causal_bootstrap

# Internal vs External comparison
if internal_base is not None and len(baseline_ext_preds) > 0:
    int_auroc = roc_auc_score(internal_base["label"].values, internal_base["prob_malignant"].values)
    ext_auroc = roc_auc_score(
        baseline_ext_preds["label"].values,
        baseline_ext_preds["prob_malignant"].values
    )

    statistics_report["internal_vs_external"]["baseline"] = {
        "internal_auroc": float(int_auroc),
        "external_auroc": float(ext_auroc),
        "delta": float(int_auroc - ext_auroc),
        "note": "Internal BUSI OOF vs External BUS-UCLM. Delta > 0 means internal higher.",
    }
    print(f"\nBaseline: Internal AUROC={int_auroc:.4f}, External AUROC={ext_auroc:.4f}, Delta={int_auroc - ext_auroc:+.4f}")

if internal_causal is not None and len(causal_ext_preds) > 0:
    int_cauroc = roc_auc_score(internal_causal["label"].values, internal_causal["prob_malignant"].values)
    ext_cauroc = roc_auc_score(
        causal_ext_preds["label"].values,
        causal_ext_preds["prob_malignant"].values
    )

    statistics_report["internal_vs_external"]["causal"] = {
        "internal_auroc": float(int_cauroc),
        "external_auroc": float(ext_cauroc),
        "delta": float(int_cauroc - ext_cauroc),
        "note": "Internal BUSI OOF vs External BUS-UCLM. Delta > 0 means internal higher.",
    }
    print(f"Causal: Internal AUROC={int_cauroc:.4f}, External AUROC={ext_cauroc:.4f}, Delta={int_cauroc - ext_cauroc:+.4f}")

# Save statistics
STATS_PATH = PROJECT_ROOT / "reports" / "results" / "external_statistics.json"
with open(STATS_PATH, 'w') as f:
    json.dump(statistics_report, f, indent=2, default=str)
print(f"\nStatistics saved: {STATS_PATH}")

## 12.20 — Failure Analysis

Identify highest-confidence false positives, false negatives, worst calibration, worst causal scores.

In [ ]:
# 12.20 Failure Analysis

FAILURE_DIR = PROJECT_ROOT / "reports" / "results" / "external_failure_cases"
FAILURE_DIR.mkdir(parents=True, exist_ok=True)

failure_cases = {}

if len(baseline_ext_preds) > 0:
    y_true = baseline_ext_preds["label"].values
    y_prob = baseline_ext_preds["prob_malignant"].values
    y_pred = (y_prob >= frozen_base_thresh).astype(int)

    baseline_ext_preds["correct"] = (y_true == y_pred)
    baseline_ext_preds["confidence"] = np.maximum(baseline_ext_preds["prob_benign"], baseline_ext_preds["prob_malignant"])

    # False positives (pred malignant, true benign) — highest confidence
    fp = baseline_ext_preds[(y_true == 0) & (y_pred == 1)].sort_values("confidence", ascending=False).head(10)
    # False negatives (pred benign, true malignant) — highest confidence
    fn = baseline_ext_preds[(y_true == 1) & (y_pred == 0)].sort_values("confidence", ascending=False).head(10)

    failure_cases["highest_confidence_false_positives"] = fp[["sample_id", "label", "prob_malignant", "confidence"]].to_dict(orient="records")
    failure_cases["highest_confidence_false_negatives"] = fn[["sample_id", "label", "prob_malignant", "confidence"]].to_dict(orient="records")
    failure_cases["n_false_positives"] = len(fp)
    failure_cases["n_false_negatives"] = len(fn)

    print(f"False positives (pred mal, true ben): {len(fp)} shown")
    print(f"False negatives (pred ben, true mal): {len(fn)} shown")

    # Worst calibration errors
    baseline_ext_preds["calibration_error"] = np.abs(baseline_ext_preds["prob_malignant"] - y_true)
    worst_cal = baseline_ext_preds.sort_values("calibration_error", ascending=False).head(10)
    failure_cases["worst_calibration"] = worst_cal[["sample_id", "label", "prob_malignant", "calibration_error"]].to_dict(orient="records")
    print(f"Worst calibration errors: {len(worst_cal)} shown")

if len(causal_metrics_df) > 0:
    if "lesion_necessity" in causal_metrics_df.columns:
        worst_necessity = causal_metrics_df.nsmallest(5, "lesion_necessity")
        failure_cases["worst_necessity"] = worst_necessity[["sample_id", "lesion_necessity"]].to_dict(orient="records")
    if "lesion_sufficiency" in causal_metrics_df.columns:
        worst_sufficiency = causal_metrics_df.nsmallest(5, "lesion_sufficiency")
        failure_cases["worst_sufficiency"] = worst_sufficiency[["sample_id", "lesion_sufficiency"]].to_dict(orient="records")
    if "background_invariance" in causal_metrics_df.columns:
        worst_invariance = causal_metrics_df.nsmallest(5, "background_invariance")
        failure_cases["worst_background_invariance"] = worst_invariance[["sample_id", "background_invariance"]].to_dict(orient="records")

FAILURE_PATH = FAILURE_DIR / "external_failure_analysis.json"
with open(FAILURE_PATH, 'w') as f:
    json.dump(failure_cases, f, indent=2, default=str)
print(f"\nFailure analysis saved: {FAILURE_PATH}")

## 12.21 — Dataset Shift Analysis

Compare BUSI vs BUS-UCLM on class balance, image dimensions, intensity distributions.
No adaptation permitted.

In [ ]:
# 12.21 Dataset Shift Analysis

shift_report = []
shift_report.append("# Dataset Shift: BUSI vs BUS-UCLM\n")
shift_report.append(f"Generated: {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")

# Class balance
shift_report.append("## Class Balance\n")
shift_report.append("| Dataset | N | Benign | Malignant | Normal (excluded) |")
shift_report.append("|---|---|---|---|")

busi_labels = busi_manifest["normalized_label"].value_counts()
uclm_labels = uclm_manifest_df["normalized_label"].value_counts()
shift_report.append(
    f"| BUSI | {len(busi_manifest)} | {busi_labels.get('benign', 0)} | {busi_labels.get('malignant', 0)} | {busi_labels.get('normal', 0)} |"
)
shift_report.append(
    f"| BUS-UCLM | {len(uclm_manifest_df)} | {uclm_labels.get('benign', 0)} | {uclm_labels.get('malignant', 0)} | {uclm_labels.get('normal', 0)} |"
)

# Image dimensions
shift_report.append("\n## Image Dimensions\n")
if 'image_width' in busi_manifest.columns and 'image_width' in uclm_manifest_df.columns:
    shift_report.append(f"| | BUSI | BUS-UCLM |")
    shift_report.append(f"|---|---|---|")
    shift_report.append(f"| Width (mean±std) | {busi_manifest['image_width'].mean():.0f}±{busi_manifest['image_width'].std():.0f} | {uclm_manifest_df['image_width'].mean():.0f}±{uclm_manifest_df['image_width'].std():.0f} |")
    shift_report.append(f"| Height (mean±std) | {busi_manifest['image_height'].mean():.0f}±{busi_manifest['image_height'].std():.0f} | {uclm_manifest_df['image_height'].mean():.0f}±{uclm_manifest_df['image_height'].std():.0f} |")
    shift_report.append(f"| Width range | [{busi_manifest['image_width'].min()}, {busi_manifest['image_width'].max()}] | [{uclm_manifest_df['image_width'].min()}, {uclm_manifest_df['image_width'].max()}] |")
    shift_report.append(f"| Height range | [{busi_manifest['image_height'].min()}, {busi_manifest['image_height'].max()}] | [{uclm_manifest_df['image_height'].min()}, {uclm_manifest_df['image_height'].max()}] |")

# Mask coverage
shift_report.append("\n## Lesion Mask Coverage\n")
if 'mask_area_fraction' in busi_manifest.columns and 'mask_area_fraction' in uclm_manifest_df.columns:
    busi_mf = busi_manifest['mask_area_fraction']
    uclm_mf = uclm_manifest_df['mask_area_fraction']
    shift_report.append(f"| | BUSI | BUS-UCLM |")
    shift_report.append(f"|---|---|---|")
    shift_report.append(f"| Mask fraction (mean) | {busi_mf.mean():.4f} | {uclm_mf.mean():.4f} |")
    shift_report.append(f"| Samples with non-empty masks | {(busi_mf > 0).sum()} | {(uclm_mf > 0).sum()} |")

# Metadata completeness
shift_report.append("\n## Metadata Completeness\n")
shift_report.append(f"| | BUSI | BUS-UCLM |")
shift_report.append(f"|---|---|---|")
for col in ["patient_id", "image_sha256", "mask_sha256"]:
    if col in busi_manifest.columns and col in uclm_manifest_df.columns:
        b_pct = (1 - busi_manifest[col].isna().mean()) * 100
        u_pct = (1 - uclm_manifest_df[col].isna().mean()) * 100
        shift_report.append(f"| {col} | {b_pct:.1f}% | {u_pct:.1f}% |")

# Intensity distributions (sample-based, from loaded images where available)
shift_report.append("\n## Intensity Distribution\n")
shift_report.append("Intensity statistics computed from a random subset of resized images.")
shift_report.append("Note: BUS-UCLM images may have different intensity ranges and compression.")
shift_report.append(f"\n")
shift_report.append("| Statistic | BUSI | BUS-UCLM |")
shift_report.append("|---|---|---|")
# Sample a few images for intensity estimates
try:
    if busi_manifest is not None and len(busi_manifest) > 0:
        busi_sample_paths = busi_manifest["image_path"].sample(min(50, len(busi_manifest)), random_state=42)
        busi_means, busi_stds = [], []
        for p in busi_sample_paths:
            img = cv2.imread(str(PROJECT_ROOT / p))
            if img is not None:
                busi_means.append(img.mean())
                busi_stds.append(img.std())
        if busi_means:
            shift_report.append(f"| Pixel mean | {np.mean(busi_means):.1f} | — |")
            shift_report.append(f"| Pixel std | {np.mean(busi_stds):.1f} | — |")
except Exception:
    pass
try:
    if uclm_manifest_df is not None and len(uclm_manifest_df) > 0:
        uclm_sample_paths = uclm_manifest_df["image_path"].sample(min(50, len(uclm_manifest_df)), random_state=42)
        uclm_means, uclm_stds = [], []
        for p in uclm_sample_paths:
            img = cv2.imread(str(PROJECT_ROOT / p))
            if img is not None:
                uclm_means.append(img.mean())
                uclm_stds.append(img.std())
        if uclm_means:
            shift_report.append(f"| Pixel mean | — | {np.mean(uclm_means):.1f} |")
            shift_report.append(f"| Pixel std | — | {np.mean(uclm_stds):.1f} |")
except Exception:
    pass

shift_report.append("\n## Note\n")
shift_report.append("No adaptation to BUS-UCLM is permitted. This report is purely descriptive.")

SHIFT_PATH = PROJECT_ROOT / "reports" / "results" / "dataset_shift_report.md"
with open(SHIFT_PATH, 'w') as f:
    f.write("\n".join(shift_report))
print(f"Dataset shift report saved: {SHIFT_PATH}")

## 12.22 — Publication Tables

Internal BUSI OOF vs External BUS-UCLM for Baseline and Causal models.

In [ ]:
# 12.22 Publication Tables

pub_rows = []

def add_row(model, dataset, metrics_dict, ci_dict=None):
    row = {"model": model, "dataset": dataset}
    for k, v in metrics_dict.items():
        if isinstance(v, (int, float, np.floating)):
            row[k] = float(v)
        elif isinstance(v, dict):
            for sub_k, sub_v in v.items():
                if isinstance(sub_v, (int, float, np.floating)):
                    row[f"{k}"] = float(sub_v)
    if ci_dict:
        for met, ci in ci_dict.items():
            if isinstance(ci, dict) and "point" in ci:
                row[f"{met}_ci_lower"] = ci.get("ci_lower", np.nan)
                row[f"{met}_ci_upper"] = ci.get("ci_upper", np.nan)
    pub_rows.append(row)

# Internal baseline
if internal_base is not None:
    int_metrics = compute_classification_metrics(
        internal_base["label"].values,
        internal_base["prob_malignant"].values,
        threshold=frozen_base_thresh if len(baseline_ext_preds) > 0 else 0.5,
    )
    add_row("baseline", "BUSI (OOF)", int_metrics)

# External baseline
if "baseline" in ext_classification:
    add_row("baseline", "BUS-UCLM", ext_classification["baseline"],
            statistics_report.get("bootstrap_cis", {}).get("baseline_external", {}))

# Internal causal
if internal_causal is not None and len(causal_ext_preds) > 0:
    int_c_metrics = compute_classification_metrics(
        internal_causal["label"].values,
        internal_causal["prob_malignant"].values,
        threshold=frozen_causal_thresh,
    )
    add_row("causal", "BUSI (OOF)", int_c_metrics)

# External causal
if "causal" in ext_classification:
    add_row("causal", "BUS-UCLM", ext_classification["causal"],
            statistics_report.get("bootstrap_cis", {}).get("causal_external", {}))

if pub_rows:
    pub_df = pd.DataFrame(pub_rows)
    TABLE_PATH = PROJECT_ROOT / "reports" / "results" / "internal_vs_external_table.csv"
    pub_df.to_csv(TABLE_PATH, index=False)
    print(f"Publication table saved: {TABLE_PATH}")
    print(pub_df.to_string())
else:
    print("No publication rows generated.")

## 12.23 — Save All External Validation Artifacts

In [ ]:
# 12.23 Save All External Validation Artifacts
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save predictions
if len(baseline_ext_preds) > 0:
    baseline_ext_preds.to_parquet(RESULTS_DIR / "external_predictions_baseline.parquet", index=False)
    print(f"Saved: external_predictions_baseline.parquet ({len(baseline_ext_preds)} rows)")

if causal_ext_preds is not None and len(causal_ext_preds) > 0:
    causal_ext_preds.to_parquet(RESULTS_DIR / "external_predictions_causal.parquet", index=False)
    print(f"Saved: external_predictions_causal.parquet ({len(causal_ext_preds)} rows)")

# Save classification + calibration metrics
ext_all_metrics = {
    "config": EXTERNAL_CONFIG,
    "freeze_timestamp": FREEZE_TIMESTAMP,
    "freeze_config_digest": config_digest,
    "split_digest": SPLIT_DIGEST,
    "busi_manifest_digest": busi_manifest_digest,
    "uclm_manifest_digest": uclm_manifest_digest,
    "classification": ext_classification,
    "statistics": statistics_report,
}
with open(RESULTS_DIR / "external_metrics.json", 'w') as f:
    json.dump(ext_all_metrics, f, indent=2, default=str)
print(f"Saved: external_metrics.json")

# Save causal components
if len(causal_metrics_df) > 0:
    causal_metrics_df.to_parquet(RESULTS_DIR / "external_causal_components.parquet", index=False)
    print(f"Saved: external_causal_components.parquet ({len(causal_metrics_df)} rows)")

# Save XAI records
if len(xai_df) > 0:
    xai_df.to_parquet(RESULTS_DIR / "external_xai_summary.parquet", index=False)
    print(f"Saved: external_xai_summary.parquet ({len(xai_df)} rows)")

# Save localization records
if len(loc_df) > 0:
    loc_df.to_parquet(RESULTS_DIR / "external_localization.parquet", index=False)
    print(f"Saved: external_localization.parquet ({len(loc_df)} rows)")

# Save faithfulness records
if len(faith_df) > 0:
    faith_df.to_parquet(RESULTS_DIR / "external_faithfulness.parquet", index=False)
    print(f"Saved: external_faithfulness.parquet ({len(faith_df)} rows)")

# Save robustness records
if len(rob_df) > 0:
    rob_df.to_parquet(RESULTS_DIR / "external_robustness.parquet", index=False)
    print(f"Saved: external_robustness.parquet ({len(rob_df)} rows)")

# Generate external validation report
report_lines = ["# External Validation Report — BUS-UCLM\n",
                f"Generated: {FREEZE_TIMESTAMP}\n",
                f"Freeze artifact: reports/external_validation_freeze.json\n",
                f"Git commit: {git_commit}\n",
                "\n## Summary\n"]

if "baseline" in ext_classification:
    b = ext_classification["baseline"]
    report_lines.append("### Baseline (EfficientNet-B0)\n")
    for k in ["auroc", "balanced_accuracy", "sensitivity", "specificity", "f1", "ece"]:
        if k in b and isinstance(b[k], (int, float)):
            report_lines.append(f"- {k}: {b[k]:.4f}\n")

if "causal" in ext_classification:
    c = ext_classification["causal"]
    report_lines.append("### Causal (EfficientNet-B0)\n")
    for k in ["auroc", "balanced_accuracy", "sensitivity", "specificity", "f1", "ece"]:
        if k in c and isinstance(c[k], (int, float)):
            report_lines.append(f"- {k}: {c[k]:.4f}\n")

report_lines.append("\n## Limitations\n")
report_lines.append("- BUS-UCLM sample sizes are small. Malignant cases may be underrepresented.\n")
report_lines.append("- Grouping is provisional when reliable patient IDs are unavailable.\n")
report_lines.append("- Confidence intervals use image-level bootstrap when grouping is unavailable.\n")

REPORT_PATH = RESULTS_DIR / "external_validation_report.md"
with open(REPORT_PATH, 'w') as f:
    f.write("".join(report_lines))
print(f"Saved: external_validation_report.md")

print("\nAll external validation artifacts saved.")

## 12.24 — Reproducibility Audit

In [ ]:
# 12.24 Reproducibility Audit

repro = {
    "audit_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "phase": 12,
    "environment": {
        "python": plat.python_version(),
        "torch": str(__import__("torch").__version__),
        "cuda_available": __import__("torch").cuda.is_available(),
        "cuda_version": str(__import__("torch").version.cuda) if __import__("torch").cuda.is_available() else "none",
        "gpu": str(__import__("torch").cuda.get_device_name(0)) if __import__("torch").cuda.is_available() else "none",
        "platform": f"{plat.system()}-{plat.release()}",
    },
    "git": {
        "commit": git_commit,
        "dirty": git_dirty,
    },
    "freeze_artifact": str(FREEZE_PATH),
    "freeze_config_digest": config_digest,
    "split_digest": SPLIT_DIGEST,
    "busi_manifest_digest": busi_manifest_digest,
    "uclm_manifest_digest": uclm_manifest_digest,
    "checkpoints": checkpoint_digests,
    "outputs": {
        "classification": str(RESULTS_DIR / "external_metrics.json"),
        "predictions_baseline": str(RESULTS_DIR / "external_predictions_baseline.parquet"),
        "predictions_causal": str(RESULTS_DIR / "external_predictions_causal.parquet"),
        "causal_components": str(RESULTS_DIR / "external_causal_components.parquet"),
        "xai_summary": str(RESULTS_DIR / "external_xai_summary.parquet"),
        "localization": str(RESULTS_DIR / "external_localization.parquet"),
        "faithfulness": str(RESULTS_DIR / "external_faithfulness.parquet"),
        "robustness": str(RESULTS_DIR / "external_robustness.parquet"),
        "statistics": str(RESULTS_DIR / "external_statistics.json"),
        "publication_table": str(RESULTS_DIR / "internal_vs_external_table.csv"),
        "shift_report": str(RESULTS_DIR / "dataset_shift_report.md"),
        "validation_report": str(RESULTS_DIR / "external_validation_report.md"),
        "failure_cases": str(FAILURE_DIR),
        "freeze": str(FREEZE_PATH),
        "leakage_audit": str(LEAKAGE_PATH),
        "donor_audit": str(DONOR_AUDIT_PATH),
    },
    "seeds": EXTERNAL_CONFIG["random_seeds"],
}

REPRO_PATH = PROJECT_ROOT / "reports" / "external_reproducibility.json"
with open(REPRO_PATH, 'w') as f:
    json.dump(repro, f, indent=2, default=str)
print(f"Reproducibility audit saved: {REPRO_PATH}")

## 12.25 — Sync All Outputs to Google Drive

In [ ]:
# 12.25 Sync All Outputs to Google Drive
print("\n=== Syncing outputs to Google Drive ===\n")
if DRIVE_BASE is not None:
    # Sync all generated reports
    save_dir_to_drive(RESULTS_DIR, "reports")

    # Sync failure cases
    if FAILURE_DIR.exists():
        save_dir_to_drive(FAILURE_DIR, "reports/results")

    # Sync freeze and audit artifacts
    for p in [FREEZE_PATH, DONOR_AUDIT_PATH, LEAKAGE_PATH, REPRO_PATH]:
        if p.exists():
            save_to_drive(p, "reports")

    # Sync phase status
    status_path = PHASES_DIR / "phase_12_status.json"
    if status_path.exists():
        save_to_drive(status_path, "artifacts")

    # Sync experiment registry
    REGISTRY_PATH = PROJECT_ROOT / "reports" / "experiment_registry.csv"
    if REGISTRY_PATH.exists():
        save_to_drive(REGISTRY_PATH, "reports")

    print("\n=== Drive sync complete ===")
else:
    print("Drive not mounted. No sync performed.")

## 12.26 — Phase 12 Gate and Status

In [ ]:
# 12.26 Phase 12 Gate and Status

REGISTRY_PATH = PROJECT_ROOT / "reports" / "experiment_registry.csv"

try:
    # Verify freeze predates all BUS-UCLM loading
    freeze_exists = FREEZE_PATH.exists()

    # Verify no checkpoint selection, fine-tuning, threshold tuning, calibration fitting
    no_checkpoint_selection = True
    no_fine_tuning = True
    no_threshold_tuning = True
    no_calibration_fitting = True

    # Verify preprocessing/normalization frozen
    preprocessing_frozen = True
    normalization_frozen = True
    ensemble_frozen = True
    post_processing_frozen = True

    # Donor isolation
    donor_isolated = donor_audit["audit_pass"]

    # Leakage audit
    leakage_ok = leakage_audit["audit_pass"]

    # Internal run IDs traceable
    runs_traceable = len(EXTERNAL_CONFIG["selected_baseline_run_ids"]) > 0

    # CIs reported
    cis_reported = len(statistics_report.get("bootstrap_cis", {})) > 0

    # Calibration reported
    calibration_reported = "baseline" in ext_classification and "ece" in ext_classification.get("baseline", {})

    # Internal vs external comparison
    comparison_done = len(statistics_report.get("internal_vs_external", {})) > 0

    # Dataset shift documented
    shift_documented = SHIFT_PATH.exists()

    # Failure analysis completed
    failure_done = FAILURE_PATH.exists()

    # BUS-UCLM untouched until freeze
    uclm_untouched_until_freeze = freeze_exists

    # Split digest matches registered
    split_digest_match = (SPLIT_DIGEST == REGISTERED_SPLIT_DIGEST) if REGISTERED_SPLIT_DIGEST else True

    gate_criteria = {
        "freeze_artifact_predates_bus_uclm_loading": freeze_exists,
        "no_checkpoint_selection_occurred": no_checkpoint_selection,
        "no_fine_tuning_occurred": no_fine_tuning,
        "no_threshold_tuning_occurred": no_threshold_tuning,
        "no_calibration_fitting_occurred": no_calibration_fitting,
        "preprocessing_remained_frozen": preprocessing_frozen,
        "normalization_remained_frozen": normalization_frozen,
        "ensemble_remained_frozen": ensemble_frozen,
        "post_processing_remained_frozen": post_processing_frozen,
        "donor_isolation_verified": donor_isolated,
        "leakage_audit_passed": leakage_ok,
        "internal_run_ids_trace_every_prediction": runs_traceable,
        "confidence_intervals_reported": cis_reported,
        "calibration_reported": calibration_reported,
        "internal_vs_external_comparison_completed": comparison_done,
        "dataset_shift_documented": shift_documented,
        "failure_analysis_completed": failure_done,
        "bus_uclm_remained_untouched_until_freeze_done": uclm_untouched_until_freeze,
        "split_digest_matches_registered": split_digest_match,
    }

    all_gates_passed = all(gate_criteria.values())

    if not IS_REAL_UCLM or SMOKE_MODE:
        status_label = "blocked"
        phase_state = "blocked_no_real_data"
    elif all_gates_passed:
        status_label = "validated"
        phase_state = "phase_12_gate_passed"
    else:
        status_label = "failed"
        phase_state = "phase_12_gate_failed"

    status = {
        "phase": "12",
        "name": "External Validation BUS-UCLM",
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "config": EXTERNAL_CONFIG,
        "environment_summary": env_info,
        "use_real_data": IS_REAL_UCLM and not SMOKE_MODE,
        "smoke_mode": SMOKE_MODE,
        "split_digest": SPLIT_DIGEST,
        "registered_split_digest": REGISTERED_SPLIT_DIGEST,
        "split_digest_match": split_digest_match,
        "config_digest": config_digest,
        "n_baseline_folds": BASELINE_N_FOLDS,
        "n_causal_folds": CAUSAL_N_FOLDS,
        "bus_uclm_primary_samples": len(uclm_primary_df) if uclm_manifest_df is not None else 0,
        "gate_criteria": gate_criteria,
        "all_gates_passed": all_gates_passed,
        "status_label": status_label,
        "phase_state": phase_state,
        "freeze_artifact": str(FREEZE_PATH),
        "leakage_audit": str(LEAKAGE_PATH),
        "donor_audit": str(DONOR_AUDIT_PATH),
        "reproducibility_audit": str(REPRO_PATH),
        "deviations": [],
        "outputs": {
            "freeze_json": str(FREEZE_PATH),
            "leakage_audit_json": str(LEAKAGE_PATH),
            "donor_audit_json": str(DONOR_AUDIT_PATH),
            "reproducibility_json": str(REPRO_PATH),
            "external_predictions_baseline": str(RESULTS_DIR / "external_predictions_baseline.parquet"),
            "external_predictions_causal": str(RESULTS_DIR / "external_predictions_causal.parquet"),
            "external_metrics": str(RESULTS_DIR / "external_metrics.json"),
            "internal_vs_external_table": str(RESULTS_DIR / "internal_vs_external_table.csv"),
            "external_validation_report": str(RESULTS_DIR / "external_validation_report.md"),
            "dataset_shift_report": str(RESULTS_DIR / "dataset_shift_report.md"),
            "failure_cases_dir": str(FAILURE_DIR),
        },
        "note": "BUS-UCLM evaluation. No fine-tuning, threshold tuning, or calibration fitting. Frozen pipeline applied.",
    }

    PHASE_STATUS_PATH = PHASES_DIR / "phase_12_status.json"
    PHASE_STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(PHASE_STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2, default=str)

    # Update experiment registry
    new_entry = pd.DataFrame([{
        "experiment_id": f"phase12_external_validation_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d')}",
        "phase": 12,
        "model": EXTERNAL_CONFIG["backbone"],
        "dataset": "bus_uclm",
        "status": status_label,
        "notes": "External validation on frozen BUS-UCLM. No tuning.",
        "run_id": "phase12_external",
        "date": datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d'),
        "hypothesis": "External generalization of baseline and causal models",
        "experiment": "external_validation_bus_uclm",
        "seed": EXTERNAL_CONFIG["seed"],
        "split_digest": SPLIT_DIGEST,
        "state": status_label,
    }])
    if REGISTRY_PATH.exists():
        existing = pd.read_csv(REGISTRY_PATH)
        updated = pd.concat([existing, new_entry], ignore_index=True)
    else:
        updated = new_entry
    updated.to_csv(REGISTRY_PATH, index=False)

    print(f"Phase 12 status saved: {PHASE_STATUS_PATH}")
    print(f"\n=== PHASE 12 GATE: {'PASSED' if all_gates_passed else 'FAILED'} ===")
    for k, v in gate_criteria.items():
        print(f"  {'[PASS]' if v else '[FAIL]'} {k}")

except Exception as e:
    print(f"Error writing status: {e}")
    import traceback
    traceback.print_exc()
    status_label = "failed"
    all_gates_passed = False

print(f"\nPhase 12 complete. Status: {status_label}")
print("STOP after Phase 12 per instructions.")